# Detecting Parkinson’s Disease Using Acoustic Characteristics

### Which acoustic characteristics are most useful for distinguishing Parkinson’s disease from healthy controls, and which machine learning approach best captures these differences?

**Goal:** Compare SVM, Random Forest, and k-Nearest Neighbours for PD versus healthy-control classification, and investigate informative acoustic features.


## 1. Setup
Import the required libraries and set a fixed random seed. Use a Python environment with the project dependencies and select that environment as the notebook kernel.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
)

# Using the same seed for reproducible experiments.
SEED = 42
np.random.seed(SEED)

# Change this path if the project is moved to another computer.
DATA_FOLDER = "/Users/mominah/Desktop/COMPSCI 4AL3/Project/pd-speech-detection/db"
DATA_FILE = os.path.join(DATA_FOLDER, "segment_features.csv")

# Acoustic measurements used as model inputs.
FEATURE_COLUMNS = [
    "CPPS",
    "HNR",
    "Shimmer (local)",
    "Shimmer (local_dB)",
    "LTAS Slope",
    "LTAS Tilt",
]
TARGET_COLUMN = "PD"                 # 0 = healthy control, 1 = Parkinson's disease
GROUP_COLUMN = "participant_id"      # Keeps each participant within one fold.

print("Setup ready. Python:", sys.version.split()[0])
print("Random seed:", SEED)


## 2. Load the extracted features
Load `segment_features.csv` 
Display the first few rows, column names, data types, and shape. These are saved acoustic measurements; rerunning Praat is not required.



In [ ]:
# Loading the previously extracted acoustic features.
df = pd.read_csv(DATA_FILE)

# Checking that the expected columns are present.
required_columns = [GROUP_COLUMN, "task", "segment_id", TARGET_COLUMN] + FEATURE_COLUMNS
missing_columns = [column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Inspecting the data before cleaning or standardization.
print("Loaded:", DATA_FILE)
print("Shape (segments, columns):", df.shape)
print("Unique participants:", df[GROUP_COLUMN].nunique())
print("Columns:", df.columns.tolist())
display(df.head())

print("\nData types and non-null counts:")
df.info()

print("\nSegments by class (0 = HC, 1 = PD):")
print(df[TARGET_COLUMN].value_counts().sort_index())


## 3. Check and clean the data
Keep `df` unchanged and create an unscaled copy named `clean_df`.

- Check identifiers, labels, recording tasks, and label consistency within each participant.
- Remove rows with missing, non-numeric, or infinite acoustic measurements; retain valid zeros and negative values.
- Remove exact duplicate records. Stop for review if a segment identifier has conflicting records.
- Summarize class balance, recording availability, and feature distributions.

Outliers are retained for inspection. Do not use full-dataset plots to select features or tune models. Scaling and any learned preprocessing belong inside the training folds.

**Feature units:** CPPS, HNR, local shimmer in dB, and the two LTAS band contrasts are in dB. Local shimmer is stored as a fraction (multiply by 100 only when reporting percentages). The column labelled LTAS tilt is a band-energy contrast on a trend fitted over 600–4,000 Hz.


In [ ]:
def preprocess(data):
    # Working on a copy so that the original DataFrame is preserved.
    clean = data.copy()
    segment_columns = [GROUP_COLUMN, "task", "segment_id"]

    # Missing or invalid identifiers/labels require investigation, not guessing.
    for column in [GROUP_COLUMN, "segment_id"]:
        values = pd.to_numeric(clean[column], errors="coerce")
        valid = np.isfinite(values) & (values >= 0) & (values % 1 == 0)
        if not valid.all():
            raise ValueError(f"Invalid or missing identifiers in {column}.")
        clean[column] = values.astype(int)

    if not clean[TARGET_COLUMN].isin([0, 1]).all():
        raise ValueError("PD labels must be 0 (HC) or 1 (PD), with no missing labels.")
    if not clean["task"].isin(["RT", "SD"]).all():
        raise ValueError("Recording tasks must be RT or SD, with no missing tasks.")
    clean[TARGET_COLUMN] = clean[TARGET_COLUMN].astype(int)

    # Every participant must have the same diagnosis across all their segments.
    labels_per_participant = clean.groupby(GROUP_COLUMN)[TARGET_COLUMN].nunique()
    if (labels_per_participant > 1).any():
        raise ValueError("A participant has conflicting PD labels; inspect the source data.")

    # Converting unexpected non-numeric feature values to NaN for detection.
    clean[FEATURE_COLUMNS] = clean[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")
    invalid_counts = (~np.isfinite(clean[FEATURE_COLUMNS])).sum()
    print("Invalid measurements per feature:")
    display(invalid_counts.rename("invalid_count").to_frame())

    # Removing exact copies, while flagging conflicting records for the same segment.
    before_duplicates = len(clean)
    clean = clean.drop_duplicates().copy()
    duplicate_count = before_duplicates - len(clean)
    if clean.duplicated(subset=segment_columns).any():
        raise ValueError("Conflicting records share a segment ID; inspect before proceeding.")

    # Removing rows with invalid features using one shared mask for all columns.
    valid_rows = np.isfinite(clean[FEATURE_COLUMNS]).all(axis=1)
    invalid_count = int((~valid_rows).sum())
    clean = clean.loc[valid_rows].reset_index(drop=True)
    if clean.empty:
        raise ValueError("No valid segments remain after cleaning.")
    if clean[TARGET_COLUMN].nunique() != 2:
        raise ValueError("Both PD and HC must remain after cleaning.")

    print("Original segments:", len(data))
    print("Removed exact duplicate records:", duplicate_count)
    print("Removed segments with invalid features:", invalid_count)
    print("Remaining segments:", len(clean))
    print("Remaining participants:", clean[GROUP_COLUMN].nunique())
    return clean


clean_df = preprocess(df)


In [ ]:
# Summarizing class balance at segment and participant levels.
class_names = {0: "HC", 1: "PD"}
participants = clean_df[[GROUP_COLUMN, TARGET_COLUMN]].drop_duplicates()
class_summary = pd.DataFrame({
    "participants": participants[TARGET_COLUMN].value_counts(),
    "segments": clean_df[TARGET_COLUMN].value_counts(),
}).sort_index().rename(index=class_names)
class_summary["segment_percent"] = (100 * class_summary["segments"] / len(clean_df)).round(1)
display(class_summary)

# A missing task is not a missing feature value; keep the participant's available data.
print("Segments per participant and recording task:")
recording_counts = pd.crosstab(clean_df[GROUP_COLUMN], clean_df["task"])
recording_counts = recording_counts.reindex(columns=["RT", "SD"], fill_value=0)
display(recording_counts)
print("Participants without retained segments for a task:")
display(recording_counts.eq(0).sum().rename("participant_count").to_frame())

print("Acoustic feature summary (unscaled; outliers retained):")
display(clean_df[FEATURE_COLUMNS].describe().T)


In [ ]:
# Inspecting feature distributions without removing unusual observations.
feature_units = ["dB", "dB", "fraction", "dB", "dB", "dB"]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feature, unit in zip(axes.flat, FEATURE_COLUMNS, feature_units):
    ax.hist(clean_df[feature], bins=25, edgecolor="white")
    ax.set_title(feature)
    ax.set_xlabel(f"Value ({unit})")
    ax.set_ylabel("Number of segments")
fig.suptitle("Unscaled acoustic feature distributions — all retained segments")
plt.tight_layout()
plt.show()

# Checking potentially redundant measurements; this does not select features.
print("Feature correlations (descriptive only):")
display(clean_df[FEATURE_COLUMNS].corr().round(2))


## 4. Define model inputs, labels, and participant groups
Initially use these six acoustic features:
- CPPS
- HNR
- Shimmer (local)
- Shimmer (local_dB)
- LTAS Slope
- LTAS Tilt

Define `X` as the acoustic features, `y` as `PD` (0 = HC, 1 = PD), and `groups` as `participant_id`. Keep `task` and `segment_id` as metadata; exclude them and participant IDs from model inputs.


In [ ]:
# TODO: Define FEATURE_COLUMNS and construct X, y, and groups.
# TODO: Confirm that rows remain aligned across inputs, labels, and metadata.


## 5. Define participant-separated evaluation
Plan five-fold participant-grouped cross-validation with approximate class balance. Use the same folds for every model comparison.

**All segments from a participant, including both RT and SD recordings, must stay together.** Check that training and validation participants do not overlap and that each fold contains both classes. Treat these as preliminary cross-validation results, not an independent final test.


In [ ]:
# TODO: Create reproducible participant-grouped folds.
# TODO: Check participant separation, class balance, and fold sizes.


## 6. Build preprocessing pipelines
Standardize features for SVM and k-NN inside each training fold. Random Forest can use unscaled features. If imputation or feature selection becomes necessary, fit it only on the training data within the pipeline.

Do not standardize the entire dataset before cross-validation.


In [ ]:
# TODO: Define model-specific preprocessing pipelines.
# TODO: Retain all six verified features for the initial experiments.


## 7. Train the majority-class baseline
Use a classifier that always predicts the most frequent class in its training fold. Evaluate it with the same folds and metrics as the other models.


In [ ]:
# TODO: Define and evaluate the majority-class baseline.
# TODO: Store fold metrics and out-of-fold predictions.


## 8. Train the classification models
Start with fixed, documented settings. For each model, train a fresh pipeline per fold and store fold metrics and out-of-fold predictions.

### 8.1 Support Vector Machine (SVM)


In [ ]:
# TODO: Define and evaluate an SVM pipeline using the shared folds.
# TODO: Record the kernel, regularization setting, and other parameters.


### 8.2 Random Forest


In [ ]:
# TODO: Define and evaluate a Random Forest using the shared folds.
# TODO: Record the number of trees, tree constraints, and random seed.


### 8.3 k-Nearest Neighbours (k-NN)


In [ ]:
# TODO: Define and evaluate a standardized k-NN pipeline using the shared folds.
# TODO: Record the number of neighbours and distance/weighting settings.


## 9. Compare results and evaluation metrics
Compare the baseline and three classifiers using accuracy, balanced accuracy, precision, recall (PD sensitivity), F1-score, and confusion matrices. Add ROC-AUC if suitable continuous scores are available.

Report fold means and variability. Clearly distinguish segment-level metrics from participant-level metrics. If aggregating segment predictions per participant, define the aggregation rule in advance. Discuss the limited number of independent participants.


In [ ]:
# TODO: Summarize cross-validation metrics for all models.
# TODO: Plot confusion matrices from out-of-fold predictions.
# TODO: Optionally summarize participant-level predictions separately.


## 10. Interpret acoustic feature contributions
Investigate which features influence predictions using an appropriate method, such as permutation importance on held-out folds. Consider correlated measurements when interpreting importance. Predictive importance does not establish a causal relationship with PD.


In [ ]:
# TODO: Estimate and visualize feature contributions on held-out data.
# TODO: Discuss consistency across folds and possible redundancy.


## 11. Optional improvements after the initial comparison
Consider limited hyperparameter tuning, feature-selection experiments, or RT-versus-SD comparisons. Any tuning or feature-subset selection must use inner participant-grouped validation, keeping the outer evaluation folds separate.


In [ ]:
# TODO (later): Define a small set of justified improvement experiments.
# TODO (later): Use nested participant-grouped validation if tuning settings.


## 12. Save outputs and write report notes
Save results, predictions with participant/segment identifiers, model settings, and package versions in a clearly named results folder. Summarize:
- What was implemented and the learning procedure/objective for each model.
- The evaluation strategy and preliminary results against the baseline.
- Limitations, actual TA feedback, and next steps.
- Each team member’s completed contributions.

Write conclusions only after running the experiments.


In [ ]:
# TODO: Export results, predictions, configuration, and environment versions.
# TODO: Write a short interpretation of the completed experiments.
